# M-0001 — TIRTL-seq demultiplex 前置侦察与方案锁定（v1）

状态：**Use**  
最后更新：2026-01-12

> 本 Milestone 聚焦于 **Demultiplex 策略侦察**。不包含全量运行，旨在通过小数据验证锁定后续方案。

## 1) Context / Goal

**背景**：
- 实验使用 **TIRTL-seq** 技术，数据格式为 `.fq.gz`。
- 条形码结构为 **<well code> + <plate code>** 组合。
- 选定工具：**`demultiplex` (jfjlaros)**。

**目标**：
- 在正式跑全量 demux 前，确定 Barcode 位置（Header/Sequence/Index）。
- 锁定 `demultiplex` 参数策略（Mismatch, Indel, Output format）。
- 建立最小可复现的验证环境。

## 2) Scope / Requirements

**In Scope**:
1.  **侦察工具 (Recon Tool)**：开发脚本自动分析 FASTQ Header 与 Read 结构，统计潜在 Barcode 位置。
2.  **方案文档 (Demux Plan)**：基于侦察结果，产出 `docs/demux_plan.md`，明确后续执行步骤。
3.  **模板准备**：建立 `examples/` 目录，存放 Well/Plate Barcode 参考表。
4.  **自动化验证**：实现 `scripts/verify.py`，使用合成数据验证侦察逻辑。

**Out of Scope (Non-goals)**:
-   执行全量数据的 Demultiplex。
-   下游生信分析（比对、定量等）。
-   性能优化。

**Constraints**:
-   **工具**：优先使用 `demultiplex` (jfjlaros) 生态或 Python 原生库。
-   **数据**：Repo 内严禁包含真实样本数据，仅允许合成数据或 User 本地路径引用。
-   **环境**：跨平台兼容 (Mac/Linux)。

## 3) Acceptance Criteria (AC)

| ID | Title | Criteria (Must be testable) |
| :--- | :--- | :--- |
| **AC-001** | **Repo Structure & Templates** | 1. `examples/well_barcodes.csv` 和 `examples/plate_barcodes.csv` 存在且格式说明清晰。<br>2. `scripts/` 目录存在。<br>3. `docs/demux_plan.md` 存在（初始需包含章节占位符）。 |
| **AC-002** | **Recon Tool Implementation** | 1. 脚本 `scripts/recon_fastq.py` 可执行。<br>2. 输入支持 R1/R2 (可选 I1/I2)。<br>3. **功能验证**：能输出 Header 结构分析结果（是否有 index）；能抽样输出 Reads 前 N bp 的统计信息（用于人工或自动判定 barcode 位置）。 |
| **AC-003** | **Synthetic Data Generator** | 1. 脚本 `scripts/generate_synthetic.py` (或集成在 verify 中) 可运行。<br>2. 能生成包含 `examples/` 中指定 Barcode 的微型 FASTQ (R1/R2)。<br>3. 生成的数据 Header 符合标准 Illumina 格式。 |
| **AC-004** | **Verification Workflow** | 1. `scripts/verify.py` 运行成功 (Exit Code 0)。<br>2. **流程闭环**：Verify 脚本自动生成合成数据 -> 运行 Recon Tool -> 断言 Recon 结果正确识别了合成数据的 Barcode 特征（位置/类型）。<br>3. 产出日志 `/logs/M-0001-verify-*.txt` 和摘要 `.json` (Level 2)。 |

## 4) Plan & Task Breakdown

**Task List**:

- [ ] **T-001: Initialize Repo & Templates (AC-001)**
  - Create `examples/` folder.
  - Create placeholder `well_barcodes.csv` & `plate_barcodes.csv`.
  - Create `docs/demux_plan.md` template.

- [ ] **T-002: Implement Synthetic Data Generator (AC-003)**
  - Create `scripts/generate_synthetic_data.py`.
  - Logic: Random sequence + Known Adapter/Barcode injection.

- [ ] **T-003: Implement Recon Tool (AC-002)**
  - Create `scripts/recon_fastq.py`.
  - Implement Header parsing logic.
  - Implement Seq slice frequency counting (to find constant regions/barcodes).

- [ ] **T-004: Implement Verification Script (AC-004)**
  - Create `scripts/verify.py`.
  - Integrate T-002 & T-003.
  - Add logging and JSON summary output.

- [ ] **T-005: Final Review & Doc Update (AC-001, AC-004)**
  - Run verify script.
  - Update `docs/demux_plan.md` with usage instructions for the Recon tool.

## 5) Implementation Notes

**Input Information**:
- Need to check: `TIRTL_barcode_plate.csv` (Already in workspace).
- Need input: Real FASTQ header examples (User to provide via `/examples/` or manually later, but Synthetic data suffices for Code Verification).

**Key Risks**:
- Plate code location might be variable. Recon tool needs to show "Top N most frequent sequences" in typical windows to spot it.
- Python dependencies: Keep minimal (pandas, biopython optional but allowed if necessary; try pure python if simple).

## 6) Verification

**Command**:
```bash
python3 scripts/verify.py
```

**Expected Output**:
- `build/synthetic_*.fq.gz` generated.
- `logs/M-0001-verify-<date>.txt` created.
- `logs/M-0001-verify-<date>.summary.json` created.
- Console output: "SUCCESS: Recon tool correctly identified synthetic barcodes."

## 7) Files Changed

(To be filled by Automation/User after execution)


## 8) Acceptance Summary

(To be filled by User after Verification)
- [ ] Gates passed?
- [ ] Logs checked?

## 9) Change Log

- **v1** (2026-01-07): Initial Draft.
- **v2** (2026-01-12): Refined by Antigravity. Added AC-001~004, Task T-001~005, and Verification Plan. Standardized structure.